In [3]:
"""Student: 11 QSVM, paired bootstrap CI and DeLong/Holm comparisons.
Install: pip install numpy pandas scipy scikit-learn matplotlib qiskit jinja2
Run: python eeg_qsvm_complete.py
CSV with header or ARFF; numeric and categorical features supported.
Inference assumes independent records. For repeated clients provide GROUP_COLUMN.
"""
from __future__ import annotations

import gc
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import kendalltau
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

try:
    # Qiskit 2.x
    from qiskit.circuit.library import z_feature_map, zz_feature_map

    def make_z_map(n_qubits: int, reps: int):
        return z_feature_map(feature_dimension=n_qubits, reps=reps)

    def make_zz_map(n_qubits: int, reps: int, entanglement: str):
        return zz_feature_map(
            feature_dimension=n_qubits,
            reps=reps,
            entanglement=entanglement,
        )

except ImportError:
    # Qiskit 1.x uyumluluğu
    from qiskit.circuit.library import ZFeatureMap, ZZFeatureMap

    def make_z_map(n_qubits: int, reps: int):
        return ZFeatureMap(feature_dimension=n_qubits, reps=reps)

    def make_zz_map(n_qubits: int, reps: int, entanglement: str):
        return ZZFeatureMap(
            feature_dimension=n_qubits,
            reps=reps,
            entanglement=entanglement,
        )



from scipy.stats import norm, binomtest
from scipy.io import arff
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from itertools import combinations
import json

FILE_PATH = "student.csv"
TARGET_COLUMN = "Success_GPA"  # Derived below: GPA > 2.5 -> 1, otherwise 0.
POSITIVE_LABEL = "1"  # Positive class in the source CSV.
GROUP_COLUMN = None  # Optional client/group column; never used as a feature.
DROP_COLUMNS = []
GPA_THRESHOLD = 2.5
STUDY_HOURS_THRESHOLD = 10.0
FEATURE_COLUMNS = [
    "Age", "Gender", "ParentalEducation", "StudyTimeWeekly", "Absences",
    "Tutoring", "ParentalSupport", "Sports", "Music",
]
TEST_SIZE = 0.25
MAX_SAMPLES = 500  # None: all data (quadratic kernels, much higher memory/time).
SEED = 42
C_VALUE = 1.0
FEATURE_MAP_REPS = 1
N_BOOTSTRAP = 10000
N_PERMUTATIONS = 20000
SVM_GAMMA = "scale"
AUTO_DROP_ID_COLUMNS = True
CONFIDENCE = 0.95
MAX_STATE_GB = 4.0
OUTPUT_DIR = Path("Student_GPA_SVM_QSVM_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CRY_RELATION_METHODS = ["Pearson", "Spearman", "Kendall", "Mutual Information", "Distance Correlation"]
EXPERIMENTS = [("Angle-RY", None), ("H+RY", None),
               *[("RY+CRY", m) for m in CRY_RELATION_METHODS],
               ("Z", None), ("ZZ-linear", None), ("ZZ-circular", None), ("ZZ-full", None), ("Classical-SVM-RBF", None)]

def save_table(frame, name):
    frame.to_csv(OUTPUT_DIR / (name + ".csv"), index=False)
    (OUTPUT_DIR / (name + ".tex")).write_text(frame.to_latex(index=False, float_format="%.6f", escape=True), encoding="utf-8")

def distribution(y, name):
    return [{"Split": name, "Class": i, "OriginalLabel": class_names[i],
             "Count": int((y == i).sum()), "Proportion": float((y == i).mean())} for i in (0, 1)]

path = Path(FILE_PATH)
if path.suffix.lower() == ".arff":
    records, _ = arff.loadarff(path)
    df = pd.DataFrame(records)
else:
    df = pd.read_csv(path)
df.columns = df.columns.astype(str).str.strip()
print("Input file:", path.resolve())
print("Columns:", df.columns.tolist())
for col in df.select_dtypes(include="object"):
    df[col] = df[col].map(lambda v: v.decode() if isinstance(v, bytes) else v)
# User-defined fixed feature and target transformations; no fitted thresholds.
required_columns = ["GPA", *FEATURE_COLUMNS]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing student columns: {missing_columns}")
gpa = pd.to_numeric(df["GPA"], errors="raise").replace([np.inf, -np.inf], np.nan)
# Unknown target values are removed, never imputed or classified as failures.
valid_target = gpa.notna()
print("Rows removed because GPA is missing:", int((~valid_target).sum()))
df = df.loc[valid_target].copy()
gpa = gpa.loc[valid_target]
df[TARGET_COLUMN] = (gpa > GPA_THRESHOLD).astype(int)
hours = pd.to_numeric(df["StudyTimeWeekly"], errors="raise").replace([np.inf, -np.inf], np.nan)
df["StudyTimeWeekly"] = np.where(hours.notna(), (hours >= STUDY_HOURS_THRESHOLD).astype(float), np.nan)
# Keep exactly nine predictors and the derived binary target.
# GPA, GradeClass, StudentID, Ethnicity, Extracurricular, Volunteering
# and any other columns cannot enter the feature matrix.
df = df[[*FEATURE_COLUMNS, TARGET_COLUMN]].copy()
print("Features:", FEATURE_COLUMNS)
print("Target: 1 = GPA > 2.5 (successful); 0 = GPA <= 2.5 (unsuccessful)")
print("StudyTimeWeekly: 1 = >=10 hours/week; 0 = <10 hours/week")
print(df.head().to_string(index=False))

if TARGET_COLUMN is None:
    candidates = [c for c in df.columns if "".join(ch for ch in c.lower() if ch.isalnum()) in
                  {"placement", "placementstatus", "placed", "status", "pass", "passed", "result", "outcome", "output", "class", "target", "label", "y"}]
    if len(candidates) != 1:
        raise ValueError("Set TARGET_COLUMN explicitly. Columns: " + str(df.columns.tolist()))
    TARGET_COLUMN = candidates[0]
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column not found: {TARGET_COLUMN}; columns: {df.columns.tolist()}")
df = df.dropna(subset=[TARGET_COLUMN]).reset_index(names="SourceRow")
raw_labels = df[TARGET_COLUMN].astype(str).str.strip()
# Canonicalize numeric-looking labels: 1.0 -> 1.
def canonical(v):
    try:
        f = float(v)
        return str(int(f)) if f.is_integer() else str(f)
    except ValueError:
        return str(v)
raw_labels = raw_labels.map(canonical)
positive = canonical(POSITIVE_LABEL)
unique = sorted(raw_labels.unique())
if len(unique) != 2 or positive not in unique:
    raise ValueError(f"Expected binary labels including {positive}; found {unique}")
class_names = [next(v for v in unique if v != positive), positive]
y_all = (raw_labels == positive).to_numpy(dtype=int)
if GROUP_COLUMN and df[GROUP_COLUMN].isna().any():
    raise ValueError("Missing group IDs")
all_distribution = distribution(y_all, "Full")
selection = np.arange(len(df))
if MAX_SAMPLES is not None and len(selection) > MAX_SAMPLES:
    if GROUP_COLUMN:
        raise ValueError("For grouped evaluation set MAX_SAMPLES=None to preserve groups.")
    selection, _ = train_test_split(selection, train_size=MAX_SAMPLES, stratify=y_all, random_state=SEED)
df = df.iloc[selection].reset_index(drop=True)
y = y_all[selection]
indices = np.arange(len(df))
if GROUP_COLUMN:
    train_indices, test_indices = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(indices, y, df[GROUP_COLUMN]))
else:
    train_indices, test_indices = train_test_split(indices, test_size=TEST_SIZE, stratify=y, random_state=SEED)
y_train, y_test = y[train_indices], y[test_indices]
if min(np.bincount(y_train, minlength=2)) < 5 or min(np.bincount(y_test, minlength=2)) < 2:
    raise ValueError("Need >=5 train and >=2 test observations per class.")
excluded = [TARGET_COLUMN, "SourceRow", *DROP_COLUMNS] + ([GROUP_COLUMN] if GROUP_COLUMN else [])
if AUTO_DROP_ID_COLUMNS:
    excluded += [c for c in df.columns if c != TARGET_COLUMN and
                 "".join(ch for ch in c.lower() if ch.isalnum()) in {"id", "studentid", "rollno", "rollnumber", "clientid", "customerid", "unnamed0"}]
excluded = list(dict.fromkeys(excluded))
print("Target:", TARGET_COLUMN, "Excluded:", excluded)
X = df.drop(columns=excluded).copy()
# Infer numeric/categorical types from training rows only.
# Missing markers in text columns are normalized before fitting encoders.
from sklearn.preprocessing import OrdinalEncoder
categorical_columns = []
for column in X.columns:
    series = X[column]
    if pd.api.types.is_numeric_dtype(series):
        X[column] = series.replace([np.inf, -np.inf], np.nan)
        continue
    text = series.astype("string").str.strip()
    text = text.mask(text.isin(["", "?", "NA", "N/A", "null", "None", "nan"]))
    numeric_train = pd.to_numeric(text.iloc[train_indices], errors="coerce")
    valid_train = text.iloc[train_indices].notna()
    if valid_train.any() and numeric_train[valid_train].notna().all():
        X[column] = pd.to_numeric(text, errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
    else:
        categorical_columns.append(column)
        X[column] = text.astype(object).where(text.notna(), np.nan)

if categorical_columns:
    if X.iloc[train_indices][categorical_columns].isna().all().any():
        raise ValueError("A training categorical feature contains only missing values.")
    category_imputer = SimpleImputer(strategy="most_frequent")
    category_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    train_categories = category_imputer.fit_transform(X.iloc[train_indices][categorical_columns])
    test_categories = category_imputer.transform(X.iloc[test_indices][categorical_columns])
    train_codes = category_encoder.fit_transform(train_categories)
    test_codes = category_encoder.transform(test_categories)
    mapping_rows = []
    for j, (column, categories) in enumerate(zip(categorical_columns, category_encoder.categories_)):
        codes = np.empty(len(X), dtype=float)
        codes[train_indices] = train_codes[:, j]
        codes[test_indices] = test_codes[:, j]
        X[column] = codes
        for code, category in enumerate(categories):
            mapping_rows.append(dict(Feature=column, Category=str(category), Code=code))
    save_table(pd.DataFrame(mapping_rows), "category_mappings")
    print("Categorical features (train-fitted ordinal encoding):", categorical_columns)
    print(pd.DataFrame(mapping_rows).to_string(index=False))
X = X.astype(float)
# Preserve the binary study-time feature when imputing missing hours.
study_mode = X.iloc[train_indices]["StudyTimeWeekly"].mode()
if study_mode.empty:
    raise ValueError("StudyTimeWeekly contains no observed training values.")
X["StudyTimeWeekly"] = X["StudyTimeWeekly"].fillna(float(study_mode.iloc[0]))
if X.iloc[train_indices].isna().all().any():
    raise ValueError("A training feature contains only missing values.")
selected_features = X.columns.tolist()
N_FEATURES = len(selected_features)
if N_FEATURES < 2:
    raise ValueError("ZZ comparisons require at least two features.")
estimated = len(df) * 2**N_FEATURES * 16 / 1024**3
if estimated > MAX_STATE_GB:
    raise MemoryError(f"Statevectors alone require {estimated:.2f} GiB; reduce samples/features explicitly.")
imputer = SimpleImputer(strategy="median")
X_train_df = pd.DataFrame(imputer.fit_transform(X.iloc[train_indices]), columns=selected_features)
X_test_df = pd.DataFrame(imputer.transform(X.iloc[test_indices]), columns=selected_features)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = np.clip(scaler.fit_transform(X_train_df), 0, np.pi)
X_test = np.clip(scaler.transform(X_test_df), 0, np.pi)
save_table(pd.DataFrame(all_distribution + distribution(y, "Selected") + distribution(y_train, "Train") + distribution(y_test, "Test")), "class_distributions")
split = np.full(len(df), "test", dtype=object)
split[train_indices] = "train"
manifest = pd.DataFrame({"SourceRow": df.SourceRow, "Split": split, "Label": y})
if GROUP_COLUMN:
    manifest["Group"] = df[GROUP_COLUMN]
save_table(manifest, "split_manifest")
prepared = df.copy()
prepared["Split"] = split
save_table(prepared, "selected_transformed_data")
assert selected_features == FEATURE_COLUMNS
assert not {"GPA", "GradeClass", "StudentID", "Ethnicity", "Extracurricular", "Volunteering"}.intersection(selected_features)
print(pd.read_csv(OUTPUT_DIR / "class_distributions.csv").to_string(index=False))
def distance_correlation(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    a = np.abs(x[:, None] - x[None, :])
    b = np.abs(y[:, None] - y[None, :])
    A = a - a.mean(axis=0)[None, :] - a.mean(axis=1)[:, None] + a.mean()
    B = b - b.mean(axis=0)[None, :] - b.mean(axis=1)[:, None] + b.mean()
    dcov2 = np.mean(A * B)
    dvar_x = np.mean(A * A)
    dvar_y = np.mean(B * B)
    if dvar_x <= 0.0 or dvar_y <= 0.0:
        return 0.0
    return float(np.sqrt(max(dcov2, 0.0) / np.sqrt(dvar_x * dvar_y)))


def relation_matrix(dataframe: pd.DataFrame, method: str) -> np.ndarray:
    """Simetrik feature-feature ilişki matrisi döndürür (0 ile 1 arası)."""
    n = dataframe.shape[1]
    matrix = np.zeros((n, n), dtype=float)

    if method == "Pearson":
        matrix = dataframe.corr(method="pearson").abs().to_numpy(dtype=float)

    elif method == "Spearman":
        matrix = dataframe.corr(method="spearman").abs().to_numpy(dtype=float)

    elif method == "Kendall":
        for i in range(n):
            for j in range(i + 1, n):
                value, _ = kendalltau(dataframe.iloc[:, i], dataframe.iloc[:, j])
                value = 0.0 if np.isnan(value) else abs(float(value))
                matrix[i, j] = matrix[j, i] = value

    elif method == "Mutual Information":
        for i in range(n):
            for j in range(i + 1, n):
                xi = dataframe.iloc[:, i].to_numpy().reshape(-1, 1)
                xj = dataframe.iloc[:, j].to_numpy()
                mi_ij = mutual_info_regression(
                    xi, xj, random_state=SEED
                )[0]
                mi_ji = mutual_info_regression(
                    xj.reshape(-1, 1), xi.ravel(), random_state=SEED
                )[0]
                value = float((mi_ij + mi_ji) / 2.0)
                matrix[i, j] = matrix[j, i] = value

        max_value = float(matrix.max())
        if max_value > 0.0:
            matrix /= max_value

    elif method == "Distance Correlation":
        for i in range(n):
            for j in range(i + 1, n):
                value = distance_correlation(
                    dataframe.iloc[:, i].to_numpy(),
                    dataframe.iloc[:, j].to_numpy(),
                )
                matrix[i, j] = matrix[j, i] = value

    else:
        raise ValueError(f"Bilinmeyen CRY ilişki yöntemi: {method}")

    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(matrix, 0.0)
    return np.clip(matrix, 0.0, 1.0)


def bind_library_map(feature_map: QuantumCircuit, x: np.ndarray) -> QuantumCircuit:
    parameters = list(feature_map.parameters)
    if len(parameters) != len(x):
        raise ValueError(
            f"Feature-map parametre sayısı ({len(parameters)}) ile "
            f"özellik sayısı ({len(x)}) uyuşmuyor"
        )
    return feature_map.assign_parameters(
        {parameter: float(value) for parameter, value in zip(parameters, x)},
        inplace=False,
    )


def build_circuit(
    encoding: str,
    x: np.ndarray,
    relation: np.ndarray | None = None,
) -> QuantumCircuit:
    n = len(x)

    if encoding == "Angle-RY":
        qc = QuantumCircuit(n)
        for i, value in enumerate(x):
            qc.ry(float(value), i)
        return qc

    if encoding == "H+RY":
        qc = QuantumCircuit(n)
        for i, value in enumerate(x):
            qc.h(i)
            qc.ry(float(value), i)
        return qc

    if encoding == "RY+CRY":
        if relation is None:
            raise ValueError("RY+CRY için ilişki matrisi gereklidir")

        qc = QuantumCircuit(n)
        for i, value in enumerate(x):
            qc.ry(float(value), i)

        # Korelasyon yalnızca burada kullanılır.
        # Örneğe bağlı açı, CRY katmanının fidelity kernel içinde yok olmasını önler.
        for i in range(n):
            for j in range(i + 1, n):
                theta_ij = relation[i, j] * (float(x[i]) + float(x[j])) / 2.0
                qc.cry(theta_ij, i, j)
        return qc

    if encoding == "Z":
        return bind_library_map(make_z_map(n, FEATURE_MAP_REPS), x)

    if encoding.startswith("ZZ-"):
        entanglement = encoding.split("-", maxsplit=1)[1]
        return bind_library_map(
            make_zz_map(n, FEATURE_MAP_REPS, entanglement),
            x,
        )

    raise ValueError(f"Bilinmeyen encoding: {encoding}")


def statevectors(
    X: np.ndarray,
    encoding: str,
    relation: np.ndarray | None = None,
) -> np.ndarray:
    return np.asarray(
        [
            Statevector.from_instruction(
                build_circuit(encoding, x, relation)
            ).data
            for x in X
        ],
        dtype=np.complex128,
    )


def quantum_kernel(states_a: np.ndarray, states_b: np.ndarray) -> np.ndarray:
    overlaps = states_a @ states_b.conj().T
    return np.square(np.abs(overlaps)).real


def circuit_metrics(
    encoding: str,
    sample_x: np.ndarray,
    relation: np.ndarray | None = None,
) -> dict:
    logical = build_circuit(encoding, sample_x, relation)
    logical_ops = logical.count_ops()
    decomposed = logical.decompose(reps=10)
    decomposed_ops = decomposed.count_ops()

    return {
        "LogicalH": int(logical_ops.get("h", 0)),
        "LogicalRY": int(logical_ops.get("ry", 0)),
        "LogicalCRY": int(logical_ops.get("cry", 0)),
        "LogicalCX": int(logical_ops.get("cx", 0)),
        "LogicalTotalGates": int(sum(logical_ops.values())),
        "LogicalDepth": int(logical.depth()),
        "DecomposedCX": int(decomposed_ops.get("cx", 0)),
        "DecomposedTotalGates": int(sum(decomposed_ops.values())),
        "DecomposedDepth": int(decomposed.depth()),
    }



# All models use the same train/test partition and preprocessing.
relations = {m: relation_matrix(X_train_df, m) for m in CRY_RELATION_METHODS}
for method, mat in relations.items():
    pd.DataFrame(mat, index=selected_features, columns=selected_features).to_csv(
        OUTPUT_DIR / (method.replace(' ', '_') + '_relation.csv'))


def metric_values(y, pred, prob):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    ratio = lambda a, b: a / b if b else 0.0
    return dict(TestAccuracy=ratio(tp+tn, len(y)), Precision=ratio(tp, tp+fp),
                Recall=ratio(tp, tp+fn), Specificity=ratio(tn, tn+fp),
                NPV=ratio(tn, tn+fn), F1=ratio(2*tp, 2*tp+fp+fn),
                ROC_AUC=roc_auc_score(y, prob),
                TestLogLoss=log_loss(y, np.column_stack([1-prob, prob]), labels=[0, 1]))


rows, predictions, probabilities = [], [], []
cm_rows, pred_dist = [], []
for number, (encoding, method) in enumerate(EXPERIMENTS, 1):
    label = f'RY+CRY-{method}' if method else encoding
    print(f'Experiment {number}/{len(EXPERIMENTS)}: {label}', flush=True)
    start = time.perf_counter()
    rel = relations.get(method)
    if encoding == "Classical-SVM-RBF":
        circuit_info = {k: np.nan for k in circuit_metrics("Angle-RY", X_train[0])}
        sv_time = kernel_time = 0.0
        model = SVC(kernel="rbf", C=C_VALUE, gamma=SVM_GAMMA,
                    probability=True, random_state=SEED)
        t = time.perf_counter()
        model.fit(X_train, y_train)
        fit_time = time.perf_counter()-t
        pt, pv = model.predict(X_train), model.predict(X_test)
        prob_train = model.predict_proba(X_train)[:, 1]
        prob = model.predict_proba(X_test)[:, 1]
    else:
        circuit_info = circuit_metrics(encoding, X_train[0], rel)
        t = time.perf_counter()
        st_train = statevectors(X_train, encoding, rel)
        st_test = statevectors(X_test, encoding, rel)
        sv_time = time.perf_counter()-t
        t = time.perf_counter()
        kt = quantum_kernel(st_train, st_train)
        kt = (kt + kt.T)/2
        np.fill_diagonal(kt, 1)
        kv = quantum_kernel(st_test, st_train)
        kernel_time = time.perf_counter()-t
        model = SVC(kernel='precomputed', C=C_VALUE, probability=True, random_state=SEED)
        t = time.perf_counter()
        model.fit(kt, y_train)
        fit_time = time.perf_counter()-t
        pt, pv = model.predict(kt), model.predict(kv)
        prob_train = model.predict_proba(kt)[:, 1]
        prob = model.predict_proba(kv)[:, 1]
    # AUC consistently uses positive-class probability, not hard predictions.
    values = metric_values(y_test, pv, prob)
    train_acc = accuracy_score(y_train, pt)
    rows.append(dict(Experiment=number, ExperimentLabel=label, NFeatures=N_FEATURES,
                     TrainAccuracy=train_acc, TrainLogLoss=log_loss(y_train, prob_train),
                     TrainTestGap=train_acc-values['TestAccuracy'],
                     AbsoluteGap=abs(train_acc-values['TestAccuracy']),
                     **values, **circuit_info, StatevectorTime=sv_time,
                     KernelTime=kernel_time, TrainingTime=fit_time,
                     TotalTime=time.perf_counter()-start))
    predictions.append(pv)
    probabilities.append(prob)
    pd.DataFrame({'SourceRow': df.iloc[test_indices].SourceRow.to_numpy(),
                  'y_true': y_test, 'y_pred': pv, 'p_positive': prob}).to_csv(
                      OUTPUT_DIR / f'predictions_{number:02d}.csv', index=False)
    for part, truth, pred in [('Train', y_train, pt), ('Test', y_test, pv)]:
        cm = confusion_matrix(truth, pred, labels=[0, 1])
        for i in (0, 1):
            for j in (0, 1):
                cm_rows.append(dict(Method=label, Split=part, TrueClass=i, PredictedClass=j, Count=int(cm[i, j])))
        for r in distribution(pred, part):
            pred_dist.append(dict(Method=label, **r))
        (OUTPUT_DIR / f'classification_{number:02d}_{part}.txt').write_text(
            classification_report(truth, pred, labels=[0, 1], target_names=class_names, zero_division=0), encoding='utf-8')
    print(values, flush=True)
    save_table(pd.DataFrame(rows), 'performance')  # incremental checkpoint
    if encoding != 'Classical-SVM-RBF':
        del st_train, st_test, kt, kv
    del model
    gc.collect()

results = pd.DataFrame(rows)
labels = results.ExperimentLabel.tolist()
P = np.array(probabilities)
D = np.array(predictions)
save_table(pd.DataFrame(cm_rows), 'confusion_matrices')
save_table(pd.DataFrame(pred_dist), 'predicted_class_distributions')

# Stratified paired bootstrap: identical resampled test indices for every model.
# With GROUP_COLUMN: resample entire test groups, preserving within-group dependence.
rng = np.random.default_rng(SEED)
pos, neg = np.flatnonzero(y_test == 1), np.flatnonzero(y_test == 0)
metric_names = list(metric_values(y_test, D[0], P[0]))
boot = np.empty((N_BOOTSTRAP, len(labels), len(metric_names)))
if GROUP_COLUMN:
    test_groups = df.iloc[test_indices][GROUP_COLUMN].to_numpy()
    groups = pd.unique(test_groups)
    if len(groups) < 2:
        raise ValueError('Cluster bootstrap requires at least two test groups.')
for b in range(N_BOOTSTRAP):
    if GROUP_COLUMN:
        for attempt in range(1000):
            ids = np.concatenate([np.flatnonzero(test_groups == g) for g in rng.choice(groups, len(groups), replace=True)])
            if len(np.unique(y_test[ids])) == 2:
                break
        else:
            raise ValueError('Unable to bootstrap groups containing both classes.')
    else:
        ids = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
    for j in range(len(labels)):
        boot[b, j] = list(metric_values(y_test[ids], D[j, ids], P[j, ids]).values())
    if (b+1) % 200 == 0:
        print(f'Bootstrap {b+1}/{N_BOOTSTRAP}', flush=True)
alpha = (1-CONFIDENCE)/2
ci_rows = []
for j, label in enumerate(labels):
    for k, metric in enumerate(metric_names):
        lo, hi = np.quantile(boot[:, j, k], [alpha, 1-alpha])
        ci_rows.append(dict(Method=label, Metric=metric, Estimate=results.iloc[j][metric],
                            CI_Lower=lo, CI_Upper=hi))
save_table(pd.DataFrame(ci_rows), 'metric_confidence_intervals')

# DeLong covariance with ties: placement values evaluated in bounded blocks.
# No test labels are used for model/threshold selection.
def delong_covariance(y, scores):
    positives, negatives = scores[:, y == 1], scores[:, y == 0]
    m, n = positives.shape[1], negatives.shape[1]
    vp, vn = np.empty_like(positives), np.zeros_like(negatives)
    for j in range(scores.shape[0]):
        for start in range(0, m, 64):
            a = positives[j, start:start+64, None]
            b = negatives[j, None, :]
            h = (a > b).astype(float) + 0.5*(a == b)
            vp[j, start:start+len(h)] = h.mean(axis=1)
            vn[j] += h.sum(axis=0)/m
    return np.cov(vp, ddof=1)/m + np.cov(vn, ddof=1)/n

cov = delong_covariance(y_test, P) if not GROUP_COLUMN else None
auc_index = metric_names.index('ROC_AUC')
comparisons = []
for a, b in combinations(range(len(labels)), 2):
    delta = float(results.iloc[a].ROC_AUC-results.iloc[b].ROC_AUC)
    lo, hi = np.quantile(boot[:, a, auc_index]-boot[:, b, auc_index], [alpha, 1-alpha])
    variance = max(float(cov[a,a]+cov[b,b]-2*cov[a,b]), 0) if cov is not None else np.nan
    if cov is None:
        p, status = np.nan, 'Cluster CI only; ordinary DeLong not valid for dependent rows'
    elif variance <= 1e-15:
        p = 1.0 if np.allclose(P[a], P[b], rtol=0, atol=1e-12) else np.nan
        status = 'Identical scores' if p == 1 else 'Degenerate DeLong variance; p undefined'
    else:
        p, status = float(2*norm.sf(abs(delta)/np.sqrt(variance))), 'OK'
    comparisons.append(dict(Method_A=labels[a], Method_B=labels[b], DeltaAUC=delta,
                            CI_Lower=lo, CI_Upper=hi, p_DeLong=p, Status=status))
pairs = pd.DataFrame(comparisons)
pairs['p_Holm'] = np.nan
# Retain the planned family size (all 66 comparisons), including undefined tests.
valid = pairs.p_DeLong.dropna().sort_values()
if len(valid):
    adjusted = np.minimum(1, np.maximum.accumulate(valid.to_numpy()*(len(pairs)-np.arange(len(valid)))))
    pairs.loc[valid.index, 'p_Holm'] = adjusted
pairs['Significant_Holm_005'] = pairs.p_Holm < 0.05
save_table(pairs, 'paired_auc_all_comparisons')
save_table(pairs[pairs.Method_A == 'Angle-RY'], 'paired_auc_vs_Angle_RY')

# Figures
fig, axes = plt.subplots(3, 4, figsize=(17, 12))
for j, ax in enumerate(axes.flat):
    if j >= len(labels):
        ax.axis('off'); continue
    cm = confusion_matrix(y_test, D[j], labels=[0, 1])
    ax.imshow(cm, cmap='Blues')
    for i in (0, 1):
        for k in (0, 1):
            ax.text(k, i, str(cm[i,k]), ha='center', va='center')
    ax.set(title=labels[j], xlabel='Predicted', ylabel='True', xticks=[0,1], yticks=[0,1])
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'confusion_matrices.png', dpi=220)
plt.close(fig)
for metric in ['TestAccuracy', 'F1', 'ROC_AUC']:
    k = metric_names.index(metric)
    lo, hi = np.quantile(boot[:,:,k], [alpha, 1-alpha], axis=0)
    fig, ax = plt.subplots(figsize=(11,7))
    # Draw interval endpoints directly; percentile intervals need not contain estimate.
    ax.hlines(np.arange(len(labels)), lo, hi)
    ax.plot(results[metric], np.arange(len(labels)), 'o')
    ax.set(yticks=np.arange(len(labels)), yticklabels=labels, xlabel=metric, title=f'{CONFIDENCE:.0%} bootstrap confidence intervals')
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/f'{metric}_CI.png', dpi=220); plt.close(fig)

notes = f'''Credit QSVM report
Positive class: {class_names[1]}; negative class: {class_names[0]}.
All features: {selected_features}
Same split for all 12 maps; imputation, scaling, relations fitted on training only.
Bootstrap: {N_BOOTSTRAP}; confidence: {CONFIDENCE}; seed: {SEED}.
CI method: {'whole-group percentile bootstrap' if GROUP_COLUMN else 'stratified paired percentile bootstrap of test rows'}.
CI conditions on fitted models and the selected partition; not retraining variability.
DeltaAUC = AUC(Method_A) - AUC(Method_B). Positive favors Method_A.
Pairwise CI is pointwise, not simultaneous; Holm adjusts two-sided DeLong p-values
across all 66 planned comparisons. Undefined degenerate p-values are not fabricated.
Recall, precision, F1 are binary positive-class metrics, not weighted averages.
Undefined zero-denominator precision/NPV/F1 are conventionally reported as zero.
AUC uses predict_proba[:,1]; hard predictions use SVC.predict (may disagree with 0.5).
No tuning or threshold selection on the test set. No class balancing applied.
H+RY and Angle-RY give the same fidelity kernel theoretically for this construction.
Inference assumes independent records/test rows. Repeated patients require grouped splitting and inference.
Recognized ID columns (id, studentid, rollno, rollnumber, clientid, customerid, Unnamed: 0) are excluded by default. All other non-target features are retained. Class ratios are preserved when subsampling; no balancing or class weights. Target: GPA>2.5 is 1, GPA<=2.5 is 0. StudyTimeWeekly>=10 is 1, otherwise 0. Age and Absences remain numeric. Missing study indicators are imputed on train only.
Categorical features use train-fitted ordinal encoding (one feature per source column). Category order can affect distances and relation matrices. Unknown test categories receive -1 before scaling/clipping.\nSVM uses RBF with the same training/test matrices, C and preprocessing as QSVM. No hyperparameter tuning.
'''
(OUTPUT_DIR/'methodology_notes.txt').write_text(notes, encoding='utf-8')
(OUTPUT_DIR/'configuration.json').write_text(json.dumps(dict(File=str(path), Target=TARGET_COLUMN,
    Positive=positive, Group=GROUP_COLUMN, Seed=SEED, MaxSamples=MAX_SAMPLES,
    TestSize=TEST_SIZE, Bootstrap=N_BOOTSTRAP, Confidence=CONFIDENCE,
    Features=selected_features, GPAThreshold=GPA_THRESHOLD, StudyHoursThreshold=STUDY_HOURS_THRESHOLD, CategoricalFeatures=categorical_columns, Excluded=excluded, C=C_VALUE, Gamma=SVM_GAMMA, Reps=FEATURE_MAP_REPS), indent=2), encoding='utf-8')
print(results.to_string(index=False))
print(pairs.to_string(index=False))
print(f'Completed. Outputs: {OUTPUT_DIR.resolve()}')

# Paired accuracy/F1: all 66 comparisons, with pointwise bootstrap CI.
# Direction matches AUC: Method_A minus Method_B.
def f1_batch(truth, pred):
    tp = ((pred == 1) & (truth == 1)).sum(axis=-1)
    fp = ((pred == 1) & (truth == 0)).sum(axis=-1)
    fn = ((pred == 0) & (truth == 1)).sum(axis=-1)
    den = 2*tp+fp+fn
    return np.divide(2*tp, den, out=np.zeros_like(den, dtype=float), where=den!=0)

rng_perm = np.random.default_rng(SEED+1)
accuracy_rows, f1_rows = [], []
for a, b in combinations(range(len(labels)), 2):
    ac, bc = D[a] == y_test, D[b] == y_test
    a_only = int((ac & ~bc).sum())
    b_only = int((bc & ~ac).sum())
    p_acc = (binomtest(a_only, a_only+b_only, p=0.5).pvalue
             if a_only+b_only else 1.0) if not GROUP_COLUMN else np.nan
    delta_f1 = float(f1_batch(y_test, D[a])-f1_batch(y_test, D[b]))
    if GROUP_COLUMN:
        p_f1 = np.nan
    elif np.array_equal(D[a], D[b]):
        p_f1 = 1.0
    else:
        extreme = 0
        for start in range(0, N_PERMUTATIONS, 256):
            count = min(256, N_PERMUTATIONS-start)
            swap = rng_perm.random((count, len(y_test))) < 0.5
            pa = np.where(swap, D[b], D[a])
            pb = np.where(swap, D[a], D[b])
            extreme += int((np.abs(f1_batch(y_test, pa)-f1_batch(y_test, pb)) >= abs(delta_f1)-1e-12).sum())
        p_f1 = (extreme+1)/(N_PERMUTATIONS+1)
    for metric, p_value, output in [('TestAccuracy', p_acc, accuracy_rows), ('F1', p_f1, f1_rows)]:
        k = metric_names.index(metric)
        lo, hi = np.quantile(boot[:,a,k]-boot[:,b,k], [alpha, 1-alpha])
        output.append(dict(Method_A=labels[a], Method_B=labels[b],
            Delta=float(results.iloc[a][metric]-results.iloc[b][metric]),
            CI_Lower=lo, CI_Upper=hi, p_Raw=p_value,
            A_correct_B_wrong=a_only, A_wrong_B_correct=b_only))

def holm_family(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(len(values), np.nan)
    good = np.flatnonzero(np.isfinite(values))
    order = good[np.argsort(values[good])]
    if len(order):
        adjusted[order] = np.minimum(1, np.maximum.accumulate(values[order]*(len(values)-np.arange(len(order)))))
    return adjusted

acc_pairs, f1_pairs = pd.DataFrame(accuracy_rows), pd.DataFrame(f1_rows)
for table in [acc_pairs, f1_pairs]:
    table['p_Holm'] = holm_family(table.p_Raw)
# Additional correction across all three metrics (198 planned tests).
joint = holm_family(np.concatenate([acc_pairs.p_Raw, f1_pairs.p_Raw, pairs.p_DeLong]))
n = len(pairs)
for table, values in zip([acc_pairs, f1_pairs, pairs], [joint[:n], joint[n:2*n], joint[2*n:]]):
    table['p_Holm_Joint198'] = values
for table, name in [(acc_pairs, 'accuracy'), (f1_pairs, 'f1'), (pairs, 'auc')]:
    save_table(table, 'paired_'+name+'_all')
    save_table(table[table.Method_A == 'Angle-RY'], 'paired_'+name+'_vs_Angle_RY')
    save_table(table[table.Method_B == 'Classical-SVM-RBF'], 'paired_'+name+'_vs_classical')
    print('\nPaired '+name+' comparisons (A minus B):')
    print(table.to_string(index=False))
with (OUTPUT_DIR/'methodology_notes.txt').open('a', encoding='utf-8') as file:
    file.write('\nAccuracy: exact two-sided McNemar. F1: paired prediction-swap permutation.\n')
    file.write('Each metric uses Holm over 66 tests; Joint198 additionally adjusts across all 3 metrics.\n')
    file.write('AUC/accuracy/F1 delta = Method_A minus Method_B, including QSVM minus classical.\n')
    file.write('Permutation assumes paired predictions exchangeable under the null. Grouped mode omits ordinary row-level p-values.\n')
print('All student GPA analyses complete:', OUTPUT_DIR.resolve())

for metric, table in pd.DataFrame(ci_rows).groupby("Metric", sort=False):
    print("\n"+metric+" — 95% confidence intervals")
    print(table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Input file: C:\Users\QALU\Desktop\student.csv
Columns: ['StudentID', 'Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA', 'GradeClass']
Rows removed because GPA is missing: 0
Features: ['Age', 'Gender', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Sports', 'Music']
Target: 1 = GPA > 2.5 (successful); 0 = GPA <= 2.5 (unsuccessful)
StudyTimeWeekly: 1 = >=10 hours/week; 0 = <10 hours/week
 Age  Gender  ParentalEducation  StudyTimeWeekly  Absences  Tutoring  ParentalSupport  Sports  Music  Success_GPA
  17       1                  2              1.0         7         1                2       0      1            1
  18       0                  1              1.0         0         0                1       0      0            1
  15       0                  3              0.0        26         0                2       0      0         